# Stage 4：Olist 訂單是否延遲預測

這本 Notebook 帶你完成「訂單核准時，預測最後是否超過預估送達日期」的流程。特徵與訓練邏輯由 `scripts/delay_features.py`、`scripts/train_delay_model.py` 共用，這裡負責導覽、檢查結果與示範推論。

請使用 **Shop (Python 3.12)** kernel。資料來自本機 `data/raw/olist/` 的六份 CSV，執行時不需要 MySQL。模型與大量逐筆預測保留在本機，Notebook 不內嵌這些檔案。


## 1. 先定義預測時點與答案

- **預測時點**：`order_approved_at`，也就是訂單／付款核准時間。
- **延遲標記**：實際送達的「日期」晚於預估送達的「日期」時，`delayed = 1`，其餘為 0。
- **答案可得時間**：`label_available_time` 是實際送達時間。訓練時必須檢查當時是否已知道答案。

例如預估 1 月 10 日送達，實際 1 月 10 日 23:59 送達仍算準時；1 月 11 日 00:00 送達才算延遲。Stage 3 用完整時間戳記比較午夜的統計方式與這裡不同，因此延遲率可能不同。

歷史訓練資料限於已送達、所需時間完整且時序合理的訂單。這個模型預測的是這群訂單的配送延遲，不包含取消或始終未送達的風險。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import joblib
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "scripts" / "delay_features.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("請從專案根目錄或 notebooks 資料夾執行這本 Notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.delay_features import (
    CATEGORICAL_FEATURES, FEATURES, NUMERIC_FEATURES, build_delay_dataset,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
MODEL_PATH = PROJECT_ROOT / "models" / "olist_delay_model.joblib"
METRICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "metrics.json"

print("專案根目錄：", PROJECT_ROOT)
print("Python：", sys.executable)


## 2. 建立每訂單一列的資料

商品明細與付款紀錄會各自彙總，再合併到訂單，避免一張訂單有多個商品與多筆付款時把金額重複加總。下面只顯示資料量、延遲比例與特徵摘要。


In [ ]:
dataset = build_delay_dataset(RAW_DIR)

display(pd.DataFrame([{
    "合格訂單數": dataset.shape[0],
    "資料欄位數": dataset.shape[1],
    "模型特徵數": len(FEATURES),
    "延遲訂單數": int(dataset["delayed"].sum()),
    "延遲比例": float(dataset["delayed"].mean()),
    "最早核准時間": dataset["prediction_time"].min(),
    "最晚核准時間": dataset["prediction_time"].max(),
}]))


## 3. 核准時可以知道哪些特徵

數值特徵包含商品數、商品與運費總額、付款資訊、商品重量與體積、預估配送天數、核准等待時間、購買時段及跨州商品比例。類別特徵包含顧客州別、主要賣家州別、主要品類與付款方式。完整欄位使用明確白名單 `FEATURES`。

實際送達、交給物流的時間、評論、最終訂單狀態，以及訂單／顧客 ID 都不進入模型。前幾項屬於預測之後才知道的資訊，ID 也不作為預測特徵。缺值補齊、標準化與類別編碼都放在 Pipeline，只用訓練集學習。

Olist 是事後快照資料，沒有每次商品、付款或預估交期變更的完整時間紀錄。本階段假設這些欄位在核准時已知；未來接真實系統時，應保存核准當下的快照來驗證這個假設。


In [ ]:
feature_summary = pd.DataFrame({
    "feature": FEATURES,
    "kind": ["numeric" if name in NUMERIC_FEATURES else "categorical"
             for name in FEATURES],
    "missing_rate": [float(dataset[name].isna().mean()) for name in FEATURES],
    "distinct_values": [dataset[name].nunique(dropna=True) for name in FEATURES],
})
display(feature_summary)


## 4. 依時間切分，排除當時還不知道的答案

訓練程式依 `prediction_time` 排序，以約 60%／20%／20% 的訂單量建立 train、validation、test；同一天的訂單全部放在同一區間，邊界日歸到後面的區間。

只做時間切分仍可能用到未來答案，因此還會做 **label maturity purge**：

- train 的送達時間必須嚴格早於 validation 開始時間。
- validation 的送達時間必須嚴格早於 test 開始時間。

因此三組實際筆數會受到日期邊界與 purge 影響。這次保留選定的原始 train 模型，不在選模後合併 validation 重訓；模型、前處理與門檻先固定，再對 test 預測一次。


## 5. 使用現有訓練成果，必要時由 CLI 建立

`RUN_TRAINING = False` 時，模型與報告都存在就直接讀取；任一檔案不存在時會自動執行訓練。若要重新產生整套成果，把設定改為 `True`。

CLI 使用目前 kernel 的 `sys.executable`，確保與 Notebook 使用同一個 Python 環境。


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING or not MODEL_PATH.is_file() or not METRICS_PATH.is_file():
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "train_delay_model.py"),
        "--data-dir", str(RAW_DIR),
        "--output-dir", str(METRICS_PATH.parent),
        "--model-path", str(MODEL_PATH),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("使用已存在的模型與 metrics.json。")


In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
artifact = joblib.load(MODEL_PATH)

if artifact["feature_names"] != list(FEATURES):
    raise ValueError("模型特徵與目前程式不同，請重新訓練。")
if artifact["selected_model"] != metrics["selected_model"]:
    raise ValueError("模型與報告不一致，請重新產生同一批訓練成果。")

split_info = metrics["metadata"]["split"]
print("Validation 開始：", split_info["validation_start"])
print("Test 開始：", split_info["test_start"])
display(pd.DataFrame.from_dict(split_info["partitions"], orient="index"))


## 6. 在 validation 選模型與決策門檻

這次比較 Dummy baseline、Logistic Regression、Random Forest。Dummy 只作為基本參考；可選模型是 Logistic Regression 與 Random Forest，分別學習線性關係與樹模型的非線性關係。

先在兩個可選模型中，以 validation 的 **Average Precision（AP）最高者選模型**，並檢查 AP 是否高於 Dummy。AP 衡量排序能力，在延遲訂單較少的資料中特別有參考價值。再只使用被選模型的 validation 分數，選擇 **F1 最高的門檻**。F1 同時考慮 precision 與 recall；這是本階段的實驗策略，未來可依人工查核量或延遲漏判成本調整。

下表比較候選模型的 validation 結果。這些分類指標使用 0.5 門檻；AP 和 ROC AUC 使用連續分數，不受分類門檻影響。


In [ ]:
validation_rows = []
for model_name, result in metrics["model_comparison"].items():
    row = {"model": model_name, "status": result["status"]}
    if result["status"] == "ok":
        row.update(result["validation_at_0_5"])
        row["fit_seconds"] = result["fit_seconds"]
    else:
        row["error"] = result.get("error", "")
    validation_rows.append(row)

models_validation = pd.DataFrame(validation_rows)
display(models_validation)


In [ ]:
selected = pd.DataFrame([{
    "selected_model": artifact["selected_model"],
    "threshold": float(artifact["threshold"]),
    "selection_metric": "validation average_precision",
    "threshold_objective": "validation F1",
    "validation_AP": metrics["validation"]["average_precision"],
    "validation_AP_above_dummy": metrics["metadata"]["model_selection"]["validation_ap_above_dummy"],
    "validation_F1_at_selected_threshold": metrics["validation"]["f1"],
}])
display(selected)


## 7. 最後才看 test，記錄保留資料的表現

Test 不參與補值、編碼、模型選擇或門檻選擇。下面讀取凍結 Pipeline 在 test 的一次評估結果，並與全猜準時的固定 baseline 比較，不使用 test 結果反覆調參。

- **Precision**：判成延遲的訂單中，實際延遲的比例。
- **Recall**：實際延遲的訂單中，被模型抓到的比例。
- **F1**：precision 與 recall 的調和平均。
- **AP／ROC AUC**：衡量連續分數的排序能力。
- **Accuracy／balanced accuracy**：作為補充，需一起考慮延遲與準時的比例。


In [ ]:
test_keys = [
    "rows", "positive_rate", "threshold", "average_precision", "roc_auc",
    "f1", "recall", "precision", "balanced_accuracy", "accuracy",
]
testmetrics = pd.DataFrame([
    {"model": artifact["selected_model"],
     **{key: metrics["test"][key] for key in test_keys}},
    {"model": "all_on_time_baseline",
     **{key: metrics["test_baseline"][key] for key in test_keys}},
])
display(testmetrics)

confusion = pd.DataFrame(
    metrics["test"]["confusion_matrix"],
    index=["實際準時", "實際延遲"],
    columns=["預測準時", "預測延遲"],
)
display(confusion)
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_test_confusion_matrix.png"
)))


## 8. 看哪些特徵影響 validation 的排序

Permutation importance 會打亂某一欄，再看 validation AP 下降多少。下降越多，代表被選模型較依賴該欄；負值或接近零代表這次抽樣沒有顯示穩定的正面貢獻。

圖表使用 validation 抽樣，未用 test 來挑特徵。這是模型行為的描述，不能當成延遲的因果證據；高度相關的欄位也可能分攤重要性。


In [ ]:
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_validation_permutation_importance.png"
)))


## 9. 用一筆資料示範 Pipeline 推論

這裡取歷史資料的一筆來示範介面，只把白名單特徵交給儲存的 Pipeline。前處理會沿用訓練時學到的設定；決策門檻沿用 artifact 內 validation 選出的值。

`predict_proba` 輸出作為延遲分數使用。這次沒有做機率校準，分數不應直接解讀成已校準的真實延遲機率。這一筆示範也不構成額外的效能評估。


In [ ]:
example = dataset.iloc[[0]]
feature_row = example.loc[:, artifact["feature_names"]].copy()
pipeline = artifact["pipeline"]
positive_class_index = list(pipeline.classes_).index(1)
delay_score = float(pipeline.predict_proba(feature_row)[0, positive_class_index])
threshold = float(artifact["threshold"])

display(pd.DataFrame([{
    "order_id": example["order_id"].iloc[0],
    "model": artifact["selected_model"],
    "delay_score_uncalibrated": delay_score,
    "threshold": threshold,
    "predicted_delayed": int(delay_score >= threshold),
}]))


## 10. 可選：比較更多模型的開發區間時間回測

新的比較加入 Extra Trees、HistGradient Boosting、XGBoost、LightGBM，連同 Dummy、Logistic Regression 與 Random Forest，一共比較七種模型。報告放在 `reports/stage4/model_comparison.json`，這個區塊只讀取既有成果，缺檔時會顯示執行指令。

這份報告使用三個**開發區間回測**，各自建立 train、validation、evaluation。Validation 與 evaluation 之間留一個月等待區間，讓 validation 的送達標記成熟；等待區間新核准的訂單不加入 validation 或 evaluation。Train 的標記必須在 validation 開始前已知，validation 的標記則必須在 evaluation 開始前已知。各模型在 validation 選門檻後，才評估該 fold 的 evaluation；Dummy 保持固定 0.5 門檻作為參考。

回測的 evaluation 只包含原始固定 test 開始時間（2018-05-26）之前核准的訂單，但保留這些訂單事後才取得的送達答案。因此它是事後整理的歷史評估，並非 evaluation 月底當時就能知道全部結果。

**原先 Random Forest 的固定 test 結果與模型 artifact 保持原樣。** 回測平均 AP 比較反映開發區間的表現，與前面固定 test 的單次評估不同，不能把兩者的指標直接當成同一批訂單的結果。


In [ ]:
COMPARISON_PATH = PROJECT_ROOT / "reports" / "stage4" / "model_comparison.json"
COMPARISON_IMAGE = (
    PROJECT_ROOT / "reports" / "images" / "stage4_backtest_model_comparison.png"
)

if COMPARISON_PATH.is_file():
    comparison_report = json.loads(COMPARISON_PATH.read_text(encoding="utf-8"))
    print("開發區間時間回測報告：", comparison_report["status"])
    comparison_table = pd.json_normalize(comparison_report["summary"])
    comparison_columns = {
        "display_name": "model",
        "status": "status",
        "successful_folds": "successful_folds",
        "development_ap_rank": "development_AP_rank",
        "metrics.average_precision.mean": "AP_mean",
        "metrics.average_precision.std": "AP_std",
        "metrics.roc_auc.mean": "ROC_AUC_mean",
        "metrics.f1.mean": "F1_mean",
        "metrics.recall.mean": "recall_mean",
        "metrics.precision.mean": "precision_mean",
        "metrics.ap_lift_over_prevalence.mean": "AP_lift_over_prevalence_mean",
    }
    display(comparison_table.reindex(columns=comparison_columns).rename(
        columns=comparison_columns
    ))
    if COMPARISON_IMAGE.is_file():
        display(Image(filename=str(COMPARISON_IMAGE)))
else:
    compare_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "compare_delay_models.py"),
    ]
    print("尚未產生額外模型比較；如要執行，請在終端機使用：")
    print(subprocess.list2cmdline(compare_command))


## 11. 減少誤報：平衡模式與歷史特徵實驗

使用者選定的目標是：validation 至少抓到 30% 延遲訂單，再盡量減少提醒量。新版比較 RF／Logistic、原 21 欄／加入 10 個過去 90 天已送達特徵，所有選模與門檻選擇都只用 validation。

歷史窗口為 `[核准時間−90天, 核准時間)`，答案必須當時已知，同 timestamp 或未來結果不能使用。結果會隨已送達事件更新，但模型不在 evaluation 重訓。

本次歷史特徵沒有勝出，最終平衡 artifact 仍為 static RF。逐月自動選模雖降低合計誤報，卻有一個月召回僅 1.12%，不能只看合計。因此保留原版預設；要使用新門檻，執行 `predict_delay.py --profile balanced`。

下方合計 precision／recall 由加總 TP／FP／FN 計算，與前面每月指標的平均不同。最終保留期間已檢視過，新結果僅是回溯診斷，不是新的獨立測試。缺檔時只顯示指令，不自動重新訓練。

In [ ]:
ALERT_PATH = PROJECT_ROOT / "reports" / "stage4" / "alert_optimization.json"
if ALERT_PATH.is_file():
    alert_report = json.loads(ALERT_PATH.read_text(encoding="utf-8"))
    print("Validation 最低召回目標：", alert_report["min_validation_recall"])
    display(pd.DataFrame(alert_report["summary"])[[
        "model", "policy", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "false_positive_reduction_vs_rf_f1",
    ]])
    display(pd.DataFrame([{
        "month": fold["name"],
        "selected_model": fold["experiment"]["selected_model"],
        "evaluation_recall": fold["experiment"]["models"][fold["experiment"]["selected_model"]]["evaluation"]["balanced"]["recall"],
    } for fold in alert_report["folds"]]))
    final_experiment = alert_report["final"]["experiment"]
    final_name = final_experiment["selected_model"]
    final_model = final_experiment["models"][final_name]
    print("最終候選：", final_name, "門檻：", final_model["thresholds"]["balanced"])
    print("以下為已檢視保留資料的回溯診斷，非新的獨立測試。")
    display(pd.DataFrame([{
        "profile": label, "FP": metrics["confusion_matrix"][0][1],
        "TP": metrics["confusion_matrix"][1][1], "FN": metrics["confusion_matrix"][1][0],
        "precision": metrics["precision"], "recall": metrics["recall"],
    } for label, metrics in (
        ("original", final_experiment["models"]["random_forest_static"]["evaluation"]["f1"]),
        ("balanced", final_model["evaluation"]["balanced"]),
    )]))
    alert_image = PROJECT_ROOT / "reports" / "images" / "stage4_alert_tradeoff.png"
    if alert_image.is_file():
        display(Image(filename=str(alert_image)))
else:
    print("請先執行：", subprocess.list2cmdline([
        sys.executable, str(PROJECT_ROOT / "scripts" / "improve_delay_alerts.py"),
        "--min-recall", "0.30", "--n-jobs", "4",
    ]))


## 12. 可選：月份落差與誤報診斷

這裡讀取 `reports/stage4/monthly_error_diagnostics.json`，回溯檢查已看過的 holdout。診斷使用儲存的平衡模型與**固定決策門檻**，各月都沿用同一門檻；沒有重新訓練、依月份調門檻，或改善模型。這些結果屬於既有保留資料的事後分析，不是新的獨立測試。

月表列出誤報 FP、抓到的延遲 TP、漏掉的延遲 FN 與召回率。**FPR 的分母是實際準時訂單數**，不是提醒數；`partial_month = True` 表示只涵蓋部分月份，不能直接拿件數和完整月比較。

路線表先保留 `sufficient_on_time_rows` 為真的群組，再依 FP 件數取前五名，同時列出準時訂單數與提醒數。即使通過最低樣本量篩選，也要留意分母與小樣本波動。承諾天數使用核准到預估送達的曆日差，實際天數使用核准到送達的連續天數，兩者是描述統計，不能直接替代曆日延遲標記。

這些分組與月份差異描述的是關聯，不能據此判定州別、路線或交期造成誤報。報告缺少時，下方只會顯示產生診斷的指令。


In [ ]:
DIAGNOSTICS_PATH = (
    PROJECT_ROOT / "reports" / "stage4" / "monthly_error_diagnostics.json"
)
if DIAGNOSTICS_PATH.is_file():
    diagnostic_report = json.loads(DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
    print("回溯診斷模型：", diagnostic_report["model"]["selected_model"])
    print("固定門檻：", diagnostic_report["model"]["threshold"])

    month_columns = [
        "month", "rows", "on_time_rows", "false_positives", "false_positive_rate",
        "true_positives", "false_negatives", "recall", "partial_month",
    ]
    monthly_errors = pd.DataFrame(diagnostic_report["test_months"])
    display(monthly_errors[month_columns].rename(columns={
        "false_positives": "FP", "false_positive_rate": "FPR",
        "true_positives": "TP", "false_negatives": "FN",
    }))

    print("準時樣本數足夠的路線：依 FP 件數列前五名")
    route_table = pd.DataFrame(diagnostic_report["groups"]["state_route"])
    if route_table.empty:
        print("沒有可列出的路線群組。")
    else:
        route_top5 = route_table.loc[
            route_table["sufficient_on_time_rows"].eq(True)
        ].sort_values(
            ["false_positives", "false_positive_rate"], ascending=False
        ).head(5)
        display(route_top5[[
            "state_route", "rows", "on_time_rows", "alert_rows",
            "false_positives", "false_positive_rate", "precision", "recall",
        ]].rename(columns={"false_positives": "FP", "false_positive_rate": "FPR"}))

    print("各月承諾與實際送達天數：")
    delivery_days_summary = pd.DataFrame([{
        "month": row["month"],
        "rows": row["rows"],
        "promise_days_mean": row["feature_and_outcome_summary"]["promise_days_mean"],
        "actual_delivery_days_mean": row["feature_and_outcome_summary"]["actual_delivery_days_mean"],
        "partial_month": row["partial_month"],
    } for row in diagnostic_report["test_months"]])
    display(delivery_days_summary)

    for image_name in (
        "stage4_monthly_errors.png", "stage4_false_positive_groups.png"
    ):
        diagnostic_image = PROJECT_ROOT / "reports" / "images" / image_name
        if diagnostic_image.is_file():
            display(Image(filename=str(diagnostic_image)))
else:
    print("尚未產生月份診斷；請在終端機執行：")
    print(subprocess.list2cmdline([
        str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"),
        str(PROJECT_ROOT / "scripts" / "diagnose_delay_errors.py"),
    ]))


## 13. 可選：等待期與路線承諾特徵實驗

這裡只讀取 `reports/stage4/waiting_strategy_comparison.json` 與既有圖表，不會啟動訓練。**60 天是看結果前指定的主要等待期，30 天是敏感度分析**，不是看 evaluation 成績後挑較好的等待期。每個等待期比較 Random Forest／Logistic Regression、原始特徵／路線承諾特徵，另列 validation 自動選模策略，共 10 種策略。

以 evaluation 開始日 E 為例，validation 核准區間為 `[E−等待天數−30天, E−等待天數)`；train 的核准截止日再比 validation 開始早一個等待期。Train 標籤必須嚴格早於 validation 開始，validation 標籤必須嚴格早於 E。**等待期改變時，train 與 validation 日期、訓練量及資料新近程度也一起改變**，因此比較結果不能解釋成單純延長等待期的因果效果。

路線特徵以主賣家州別 → 顧客州別區分方向；90 天歷史只用核准當時已知的有效送達事件。送達天數分位數與承諾天數都採**曆日尺度**。歷史會隨時間經過及送達結果取得而動態更新，但各折模型不在 evaluation 重新訓練。有效已送達 cohort 仍不包含所有未送達、取消或其他無效訂單，不能把成熟標籤保留率當成所有核准訂單的結果涵蓋率。

所有模型與門檻只用 train／validation 決定；**30% recall 是 validation 的最低目標，不保證後續月份達標**。下方逐月列出 validation 延遲正例保留率與被排除正例數，合計 precision／recall 則由加總 TP／FP／FN 計算，並另列最差月份 recall，避免只看合計。

這三個開發月份已在先前實驗檢視過，屬於開發回測，不是新的未知 test。**本次實驗不評估 2018-05-26 起的原 holdout**。研究 artifact 另存於 `models/research/`，未替換原版或平衡模式的推論預設。


In [ ]:
WAITING_PATH = (
    PROJECT_ROOT / "reports" / "stage4" / "waiting_strategy_comparison.json"
)
if WAITING_PATH.is_file():
    waiting_report = json.loads(WAITING_PATH.read_text(encoding="utf-8"))
    print("預先指定的主要等待期（天）：", waiting_report["primary_wait_days_prespecified"])
    print("Validation 最低召回目標：", waiting_report["min_validation_recall"])
    print("以下為已檢視開發月份的回測；本次不評估原 holdout，也不替換推論預設。")

    waiting_summary = pd.DataFrame(waiting_report["summary"])
    display(waiting_summary[[
        "wait_days", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_month_recall",
    ]].rename(columns={
        "false_positives": "FP", "true_positives": "TP", "false_negatives": "FN",
    }))

    waiting_fold_rows = []
    for fold in waiting_report["folds"]:
        split = fold["split"]
        validation = split["partitions"]["validation"]
        waiting_fold_rows.append({
            "wait_days": fold["wait_days"],
            "month": fold["name"],
            "selected_model": fold["experiment"]["selected_model"],
            "train_prediction_end": split["train_prediction_end"],
            "validation_start": split["validation_start"],
            "validation_end": split["validation_end"],
            "validation_positive_rows_before_purge": validation["positive_rows_before_purge"],
            "validation_positive_retention_rate": validation["positive_retention_rate"],
            "validation_purged_positive_rows": validation["purged_positive_rows"],
        })
    print("有效已送達 cohort 內的標籤成熟情況；非所有核准訂單涵蓋率：")
    display(pd.DataFrame(waiting_fold_rows))

    waiting_image = PROJECT_ROOT / "reports" / "images" / "stage4_waiting_tradeoff.png"
    if waiting_image.is_file():
        display(Image(filename=str(waiting_image)))
else:
    print("python scripts/compare_waiting_strategies.py --n-jobs 4")


## 14. 可選：多月份驗證

本節只讀取 `reports/stage4/multimonth_validation.json`，不啟動訓練。固定至少 60 天等待，取連續三個完整日曆月作 validation；同一模型在最早驗證月開始時訓練，之後固定。訓練核准截止再早 60 天，訓練標籤嚴格早於模型訓練日，驗證標籤嚴格早於 evaluation 開始日。

預先指定的主策略 `every_month_floor` 使用同一門檻，要求每個驗證月份 recall ≥30%，再選提醒量最少的候選。共同門檻是各月最高可行門檻的最小值。對照 `pooled_3months` 只要求三月合計達標；兩者使用完全相同 train、模型、分數及全三月選模母體。`recent_month` 只用最新驗證月決定門檻與選模，作描述性參考。

擴大 validation 會讓訓練資料更舊、筆數更少，舊單月實驗只作背景。各 fold 可能重用歷史月份，早期 evaluation 在之後 fold 成為 validation；這是滾動開發回測，不是獨立未知測試。本輪仍不評估 2018-05-26 起的原 holdout。**每個驗證月達標也不保證後續月份達標**；研究模型另存，不自動替換預設。


In [ ]:
MULTIMONTH_PATH = PROJECT_ROOT / "reports" / "stage4" / "multimonth_validation.json"
if MULTIMONTH_PATH.is_file():
    multimonth_report = json.loads(MULTIMONTH_PATH.read_text(encoding="utf-8"))
    print("主策略：", multimonth_report["primary_strategy_prespecified"])
    print("已檢視開發月份回測；每月驗證達標不保證後續達標。")
    display(pd.DataFrame(multimonth_report["summary"])[[
        "strategy", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_evaluation_month_recall",
        "minimum_validation_month_recall",
    ]])
    multimonth_selections, multimonth_coverage = [], []
    for fold in multimonth_report["folds"]:
        for strategy, candidate in fold["experiment"]["selected_models"].items():
            decision = fold["experiment"]["models"][candidate]["strategies"][strategy]
            multimonth_selections.append({
                "evaluation_month": fold["name"], "strategy": strategy, "model": candidate,
                "threshold": decision["threshold"],
                "validation_month_recalls": {
                    month: metrics["recall"] for month, metrics in decision["validation_by_month"].items()
                },
                "evaluation_recall": decision["evaluation"]["recall"],
            })
        for month in fold["split"]["validation_months"]:
            multimonth_coverage.append({
                "evaluation_month": fold["name"], "validation_month": month["month"],
                "positive_rows_before_purge": month["positive_rows_before_purge"],
                "positive_rows": month["positive_rows"],
                "positive_retention_rate": month["positive_retention_rate"],
            })
    display(pd.DataFrame(multimonth_selections))
    print("標籤保留率分母限有效已送達 cohort：")
    display(pd.DataFrame(multimonth_coverage))
    multimonth_image = PROJECT_ROOT / "reports" / "images" / "stage4_multimonth_validation.png"
    if multimonth_image.is_file():
        display(Image(filename=str(multimonth_image)))
else:
    print("python scripts/compare_multimonth_validation.py --n-jobs 4")


## 15. 可選：每月重訓的滾動驗證

本節只讀取 `reports/stage4/monthly_retraining.json`，不啟動訓練。每個月 M 第一日以核准時間 `<M−60天`、結果時間 `<M` 的訂單重訓，再為 M 月訂單留下當時的 out-of-time 分數。共用的歷史月份只訓練一次；同一月份在不同 evaluation E 使用時，仍須重新按 `label_available_time < E` 篩成熟結果。

E 時依前三個成熟完整月的舊版模型分數選門檻與候選，再使用 E 當天重訓的模型預測。**最終模型可以訓練到歷史 validation 訂單，但不拿它重算這些訂單的驗證分數**。對照最新月、三月合計及預先指定的每月 recall ≥30% 主策略，原始分數跨版本共用門檻；這些分數未校準，不保證不同模型版本具有同一尺度。

逐月門檻、分數分布與後續月份的指標只用來診斷轉移效果，不依 evaluation 調整規則。本輪仍沿用已檢視的開發月份，不評估原 2018-05-26 起 holdout，也不自動提升研究模型為預設。


In [ ]:
ROLLING_PATH = PROJECT_ROOT / "reports" / "stage4" / "monthly_retraining.json"
if ROLLING_PATH.is_file():
    rolling_report = json.loads(ROLLING_PATH.read_text(encoding="utf-8"))
    print("主策略：", rolling_report["primary_strategy_prespecified"])
    print("Fit 次數：", rolling_report["fit_count"])
    display(pd.DataFrame(rolling_report["summary"])[[
        "strategy", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_evaluation_month_recall", "minimum_validation_month_recall",
    ]])
    display(pd.DataFrame([{
        "model_month": row["month"], "fit_time": row["split"]["model_fit_time"],
        "train_approval_end": row["split"]["train_prediction_end"],
        "train_rows": row["split"]["partitions"]["train"]["rows"],
        "train_positive_retention": row["split"]["partitions"]["train"]["positive_retention_rate"],
    } for row in rolling_report["monthly_training"]]))
    rolling_selections = []
    for fold in rolling_report["folds"]:
        for strategy, candidate in fold["experiment"]["selected_models"].items():
            model = fold["experiment"]["models"][candidate]
            decision = model["strategies"][strategy]
            rolling_selections.append({
                "evaluation_month": fold["name"], "strategy": strategy, "model": candidate,
                "threshold": decision["threshold"],
                "past_month_thresholds": model["score_transfer_diagnostics"]["individual_validation_month_thresholds"],
                "minimum_validation_recall": decision["minimum_validation_month_recall"],
                "evaluation_recall": decision["evaluation"]["recall"],
                "historical_validation_rows_now_in_train": fold["historical_validation_rows_in_final_train"],
            })
    display(pd.DataFrame(rolling_selections))
    print("歷史驗證分數由舊版模型產生；未以最終模型重新計算。跨版本門檻不保證後續召回。")
    rolling_image = PROJECT_ROOT / "reports" / "images" / "stage4_monthly_retraining.png"
    if rolling_image.is_file():
        display(Image(filename=str(rolling_image)))
else:
    print("python scripts/compare_monthly_retraining.py --n-jobs 4")


## 16. 可選：分數校準與每日提醒上限

本節只讀取 `reports/stage4/calibration_budget.json`，不啟動訓練。沿用既有月版模型的 OOT 分數：每個 E 的前三個歷史月，前兩月 fit 校準器，第三月選門檻與候選，三個月都只使用標籤時間 `< E` 的結果。校準與門檻在 E 決定，並非在第三個歷史月起點就部署。

比較原分數、Sigmoid、Isotonic；主方案事先指定 Sigmoid／每日最多 10%，另看 5%、20% 與無上限。每日容量為 `floor(比例 × 當日完整評分 cohort)`，未成熟標籤仍參與容量與排名，但不拿未知結果算驗證指標。同分依校準分數、原分數、訂單 ID 排序。30% 召回不可行時，保留上限並明確標示，不宣稱達標。

每日批次結束才排名，可能等待一天；歷史 cohort 仍只含有效完成訂單。推論要有完整每日批次，使用 `--complete-daily-batches` 明確承諾完整性；不能從單張訂單保證每日上限。校準指標區分 fit 描述、獨立政策驗證及開發回測，單調校準本身不能改善排序。本輪沿用已檢視的開發月份，未評分原 holdout，也未替換預設模型。


In [ ]:
CALIBRATION_PATH = PROJECT_ROOT / "reports" / "stage4" / "calibration_budget.json"
if CALIBRATION_PATH.is_file():
    calibration_report = json.loads(CALIBRATION_PATH.read_text(encoding="utf-8"))
    print("主方案：", calibration_report["primary_method_prespecified"], calibration_report["primary_budget_prespecified"])
    print("新基礎模型 fit：", calibration_report["base_model_fit_count"], "校準器 fit：", calibration_report["learned_calibrator_fit_count"])
    summary_frame = pd.DataFrame(calibration_report["summary"])
    display(summary_frame.loc[summary_frame.model.eq("validation_selected"), [
        "method", "budget", "false_positives", "true_positives", "false_negatives",
        "alerts", "precision", "recall", "minimum_evaluation_month_recall", "infeasible_validation_months",
    ]])
    calibration_quality, primary_decisions = [], []
    for fold in calibration_report["folds"]:
        for method in calibration_report["methods"]:
            metrics = fold["experiment"]["models"]["random_forest_route"][method]["probability_metrics"]
            for phase, values in metrics.items():
                calibration_quality.append({
                    "evaluation_month": fold["name"], "candidate": "random_forest_route", "method": method,
                    "phase": phase, "rows": values["rows"], "brier": values["brier_score"],
                    "log_loss": values["log_loss"], "ece": values["expected_calibration_error"],
                })
        method, budget = calibration_report["primary_method_prespecified"], calibration_report["primary_budget_prespecified"]
        candidate = fold["experiment"]["selected_models"][method][budget]
        decision = fold["experiment"]["models"][candidate][method]["strategies"][budget]
        primary_decisions.append({
            "evaluation_month": fold["name"], "candidate": candidate, "threshold": decision["threshold"],
            "validation_feasible": decision["threshold_policy"]["feasible"],
            "maximum_validation_recall": decision["threshold_policy"]["max_achievable_recall"],
            "evaluation_recall": decision["evaluation"]["recall"],
            "evaluation_alerts": decision["evaluation_batch"]["alerts"],
            "maximum_daily_alert_rate": decision["evaluation_batch"]["maximum_daily_alert_rate"],
        })
    display(pd.DataFrame(primary_decisions))
    display(pd.DataFrame(calibration_quality))
    print("10% 上限的三組歷史驗證都無法達到30%召回；減少誤報同時增加漏報。開發回測不是未知測試。")
    calibration_image = PROJECT_ROOT / "reports" / "images" / "stage4_calibration_budget.png"
    if calibration_image.is_file():
        display(Image(filename=str(calibration_image)))
else:
    print("先產生每月重訓的模型／分數，再執行 python scripts/compare_calibration_budget.py")


## 17. 可選：固定每日 10% 預算的漏報診斷與排名改善

本節只讀取 `budget_miss_diagnostics.json` 與 `budget_ranking.json`，不啟動訓練、不新增圖表。基準主方案的漏報先按路線、承諾天數與每日排名分組。絕對漏報數也受群組訂單量影響，需同時看延遲總量、召回率與小樣本標記；這些是開發資料事後診斷。

六個新候選及參數在執行前固定，包含 Route／45 欄風險特徵的 LightGBM、XGBoost，加權 LightGBM 與風險特徵 RF。每個月 M 仍只用核准 `< M−60天` 且結果 `< M` 的訂單訓練；新增的賣家、路線歷史只看各訂單核准前已送達的事件。ZIP3 是類別，並非距離。

每日完整批次排名取 `floor(10% × 當日有效評分筆數)`，門檻固定 0。三個歷史 OOT 月的未知結果仍占名額；只有結果 `< E` 的列進驗證。依最差歷史月召回、合計 TP、AP 與固定候選順序選模，固定後才計算 E 當月結果。30% 歷史下限與開發穩定度分開呈現；本輪沿用已檢視的開發月，原 holdout 未評分，不保證未知資料成績。Dashboard 依使用者要求暫緩。


In [ ]:
MISS_PATH = PROJECT_ROOT / "reports" / "stage4" / "budget_miss_diagnostics.json"
RANKING_PATH = PROJECT_ROOT / "reports" / "stage4" / "budget_ranking.json"
if MISS_PATH.is_file():
    miss_report = json.loads(MISS_PATH.read_text(encoding="utf-8"))
    baseline_groups = miss_report["diagnostics"]["aggregate"]["groups"]
    print("基準開發漏報路線；群組訂單量也會影響絕對漏報數：")
    display(pd.DataFrame(baseline_groups["state_route"]).head(10))
    display(pd.DataFrame(baseline_groups["promise_days"]))
if RANKING_PATH.is_file():
    ranking_report = json.loads(RANKING_PATH.read_text(encoding="utf-8"))
    print("每日固定預算：", ranking_report["daily_cap"])
    print("新模型 fit：", ranking_report["new_fit_count"], "沿用歷史版本：", ranking_report["reused_legacy_version_count"])
    display(pd.DataFrame(ranking_report["summary"])[[
        "model_or_selection", "selected_candidates", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    ranking_progress = []
    for fold in ranking_report["folds"]:
        candidate = fold["experiment"]["selected_models"][ranking_report["primary_selection_prespecified"]]
        decision = fold["experiment"]["models"][candidate]
        ranking_progress.append({
            "evaluation_month": fold["name"], "candidate": candidate,
            "minimum_historical_recall": decision["minimum_validation_month_recall"],
            "historical_floor_met": decision["all_validation_months_meet_30_percent"],
            "development_recall": decision["development_evaluation"]["recall"],
            "development_alerts": decision["evaluation_batch"]["alerts"],
        })
    display(pd.DataFrame(ranking_progress))
    print("適用程度：", ranking_report["suitability"])
    new_groups = ranking_report["selected_development_miss_diagnostics"]["aggregate"]["groups"]
    display(pd.DataFrame(new_groups["state_route"]).head(10))
    display(pd.DataFrame(new_groups["promise_days"]))
    print("歷史驗證選模與開發事後診斷分開；尚無新的獨立測試成績。")
else:
    print("python scripts/diagnose_budget_misses.py")
    print("python scripts/improve_budget_ranking.py --n-jobs 4")


## 18. 可選：待件負載與每日排序模型

本節只讀取第二輪 `pending_ranking.json`，不訓練、不新增圖表。完整方法見 `docs/STAGE4_PENDING_RANKING.md`。

保留第一輪 45 欄，另加全域／賣家／路線的 30、90 天待件數、30 天平均等待及兩種逾期數，共 60 欄。母體包含當時已核准的有效原始訂單，不依最終狀態過濾；送達嚴格早於查詢才移除。缺少取消事件時間，取消單可能留在窗口內，這是資料限制。

新增 Logistic、LightGBM、XGBoost 及兩種 LambdaRank，共 40 次月份訓練；前輪 80 版本重載核對。LambdaRank 以成熟訓練列的核准日分組，二元增益與 truncation 20 的 NDCG 目標只是每日 10% 召回的代理；sigmoid(margin) 是單調分數介面，不是機率校準。

仍固定每日 10% 與三個歷史月最差召回優先選模，固定後才看開發月。整體設計已參考反覆檢視的開發診斷，因此不能稱獨立測試；原 holdout 不評分。適用門檻未通過時，保留研究模型並繼續改善，Dashboard 暫緩。


In [ ]:
PENDING_PATH = PROJECT_ROOT / "reports" / "stage4" / "pending_ranking.json"
if PENDING_PATH.is_file():
    pending_report = json.loads(PENDING_PATH.read_text(encoding="utf-8"))
    print("每日上限：", pending_report["daily_cap"], "新增 fit：", pending_report["new_fit_count"])
    display(pd.DataFrame(pending_report["summary"])[[
        "model_or_selection", "selected_candidates", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    pending_progress = []
    for fold in pending_report["folds"]:
        candidate = fold["experiment"]["selected_models"][pending_report["primary_selection_prespecified"]]
        decision = fold["experiment"]["models"][candidate]
        pending_progress.append({
            "month": fold["name"], "candidate": candidate,
            "minimum_historical_recall": decision["minimum_validation_month_recall"],
            "historical_floor_met": decision["all_validation_months_meet_30_percent"],
            "development_recall": decision["development_evaluation"]["recall"],
            "development_alerts": decision["evaluation_batch"]["alerts"],
        })
    display(pd.DataFrame(pending_progress))
    print("適用程度：", pending_report["suitability"])
    pending_groups = pending_report["selected_development_miss_diagnostics"]["aggregate"]["groups"]
    display(pd.DataFrame(pending_groups["state_route"]).head(10))
    display(pd.DataFrame(pending_groups["promise_days"]))
else:
    print("python scripts/improve_pending_ranking.py --n-jobs 4")


## 19. 可選：訓練答案取得時間與等待期

本節只讀 `outcome_clock.json`，不啟動訓練、不新增圖表。比較 Logistic／原特徵、RF／路線、LightGBM／待件，交叉三種訓練設計共 72 fit：實際送達時鐘＋0 天等待、承諾到期時鐘＋0 天等待、承諾到期時鐘＋30 天等待。

只在訓練成熟篩選新增 `training_outcome_known_time = min(實際送達, 承諾日期次日午夜)`。延遲答案在逾期時即可確認，準時答案在配送時確認；原本 `label_available_time`、配送歷史、特徵、驗證成熟篩選與每日分母全部保留。九個新候選固定參數，連同前輪 15 候選依相同歷史規則選模；開發月結果不決定參數或等待期。

仍限最終有效已送達群體，不能據此推論每張原始待件將會配送；也缺少事件入庫時間與承諾變更紀錄。所有月份及整體研究設計已參考既有開發診斷，不是新的獨立測試。歷史門檻與開發穩定度分開呈現。


In [ ]:
CLOCK_PATH = PROJECT_ROOT / "reports" / "stage4" / "outcome_clock.json"
if CLOCK_PATH.is_file():
    clock_report = json.loads(CLOCK_PATH.read_text(encoding="utf-8"))
    print("每日上限：", clock_report["daily_cap"], "新增 fit：", clock_report["new_fit_count"])
    display(pd.DataFrame(clock_report["summary"])[[
        "model_or_selection", "selected_candidates", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    clock_progress = []
    for fold in clock_report["folds"]:
        candidate = fold["experiment"]["selected_models"][clock_report["primary_selection_prespecified"]]
        decision = fold["experiment"]["models"][candidate]
        clock_progress.append({
            "month": fold["name"], "candidate": candidate,
            "minimum_historical_recall": decision["minimum_validation_month_recall"],
            "historical_floor_met": decision["all_validation_months_meet_30_percent"],
            "development_recall": decision["development_evaluation"]["recall"],
        })
    display(pd.DataFrame(clock_progress))
    print("適用程度：", clock_report["suitability"])
    clock_groups = clock_report["selected_development_miss_diagnostics"]["aggregate"]["groups"]
    display(pd.DataFrame(clock_groups["state_route"]).head(10))
    display(pd.DataFrame(clock_groups["promise_days"]))
else:
    print("python scripts/improve_outcome_clock.py --n-jobs 4")


## 20. 可選：地理與組合特徵的同模型比較

本節只讀 `feature_engineering.json`，不啟動訓練、不新增圖表。四種模型以相同月份 deadline/wait0 訓練 cohort 與固定參數，比較既有 60 欄、地理＋組合特徵 83 欄，以及加出貨期限的 91 欄；LGB 的 60 欄版本重用，其餘新增 88 fit。距離與組合特徵一起加入，不能把差異全部歸因於距離。

主要選模 `primary_without_dispatch` 排除出貨期限；該欄缺少建立／修改時間，只在敏感度組比較。地理資料也沒有版本時間，假設是可於核准時查詢的靜態郵區參考；球面距離不是道路里程。新增特徵不使用訂單最終狀態或未來配送結果，歷史負載仍只用當時已知事件。

固定每日完整批次 10% 上限、三個歷史月最差召回優先選模；歷史 30% 門檻與開發月份表現分開呈現。本輪設計也受既有開發診斷啟發，所有 E 月仍是開發回測，原保留期間不重新評分。完整方法見 `docs/STAGE4_FEATURE_ENGINEERING.md`；未通過適用門檻前，維持研究模型與暫緩 Dashboard。


In [ ]:
ENGINEERING_PATH = PROJECT_ROOT / "reports" / "stage4" / "feature_engineering.json"
if ENGINEERING_PATH.is_file():
    engineering_report = json.loads(ENGINEERING_PATH.read_text(encoding="utf-8"))
    print("每日上限：", engineering_report["daily_cap"], "新增 fit：", engineering_report["new_fit_count"])
    print("主要選組：", engineering_report["primary_selection_prespecified"])
    display(pd.DataFrame(engineering_report["summary"])[[
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    paired_progress = []
    for item in engineering_report["paired_feature_comparisons"]:
        paired_progress.append({"模型": item["family"], **item["development_recall"],
            "83減60召回差": item["geo_minus_base_development_recall"],
            "91減83召回差（敏感度）": item["dispatch_minus_geo_development_recall"]})
    print("同模型、同訓練資料的開發回測；不依此表挑選部署模型：")
    display(pd.DataFrame(paired_progress))
    display(pd.DataFrame([engineering_report["suitability"]]))
else:
    print("尚無特徵工程報告；需要時執行 scripts/improve_feature_engineering.py。")


## 21. 可選：地理與負載交互消融

本節只讀 `feature_ablation.json`，不訓練、不新增圖表。保持四種模型的固定參數、deadline/wait0 訓練訂單與每日完整批次 10% 預算，比較既有 60 欄、只加地理 G 的 72 欄、只加負載交互 L 的 65 欄、G+L 的 77 欄，以及前輪完整 83 欄。

G 包含 9 個郵區位置／距離欄位與 3 個距離比例；L 包含 5 個待件壓力／逾期占比；剩餘 O 是 6 個成本、密度與日曆交互。60 欄基準已含原始待件計數、路線與 ZIP3，所以這不是移除所有地理或負載資訊。83 減 77 只是在 G+L 存在時加入 O 的差異，不是 O 單獨的效果；各差值也不是因果估計。

三個歷史 OOT 月決定選模，固定後才計算開發 E 月；這些月份與整體設計已受先前診斷影響，不是新獨立測試。地理資料缺少版本時間的限制保留，本輪不含出貨期限候選。未通過歷史逐月 30% 召回門檻前，維持研究 artifact、暫緩 Dashboard。完整方法見 `docs/STAGE4_FEATURE_ABLATION.md`。


In [ ]:
ABLATION_PATH = PROJECT_ROOT / "reports" / "stage4" / "feature_ablation.json"
if ABLATION_PATH.is_file():
    ablation_report = json.loads(ABLATION_PATH.read_text(encoding="utf-8"))
    print("固定每日上限：", ablation_report["daily_cap"], "新增 fit：", ablation_report["new_fit_count"])
    paired_rows = [{"模型": item["family"], **item["development_recall"]}
                   for item in ablation_report["paired_feature_comparisons"]]
    print("同模型、同訓練訂單的開發回測召回率；不依開發結果選模：")
    display(pd.DataFrame(paired_rows))
    ablation_summary = pd.DataFrame(ablation_report["summary"])
    display(ablation_summary.loc[ablation_summary.model_or_selection.isin(ablation_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    display(pd.DataFrame([ablation_report["suitability"]]))
else:
    print("尚無消融報告；需要時執行 scripts/ablate_delay_features.py。")


## 22. 可選：Logistic 負載組 L2 正則化強度

本節只讀 `logistic_regularization.json`，不啟動訓練、不新增圖表。固定前輪負載組 65 欄與訓練 cohort，比較 C=0.01、0.03、0.1、0.3、1、3、10；C 越小代表 L2 正則化越強。七格除了 C，模型及前處理參數完全相同；所有格統一 max_iter=10000 並確認收斂，避免迭代上限干擾比較。

七格全部重訓，共 56 個新月份版本。新 C=1 必須重現前輪已收斂、max_iter=2000 的 C=1：核對分數、係數、截距及實際迭代次數。迭代次數與係數範數是求解／模型診斷，不拿來代替召回率或選模。

仍以三個歷史 OOT 月的最差召回、合計 TP、AP、固定候選順序選模，固定每日完整批次 10% 上限；E 月只作開發觀察。本輪設計已受先前開發結果啟發，不是獨立測試；原保留期間不重評，未達歷史每月 30% 召回前不自動換預設模型、Dashboard 暫緩。完整方法見 `docs/STAGE4_LOGISTIC_REGULARIZATION.md`。


In [ ]:
REGULARIZATION_PATH = PROJECT_ROOT / "reports" / "stage4" / "logistic_regularization.json"
if REGULARIZATION_PATH.is_file():
    regularization_report = json.loads(REGULARIZATION_PATH.read_text(encoding="utf-8"))
    print("每日固定上限：", regularization_report["daily_cap"], "新增 fit：", regularization_report["new_fit_count"])
    print("各 C 的開發回測；不依此表挑部署參數：")
    display(pd.DataFrame(regularization_report["regularization_comparisons"])[[
        "C", "development_true_positives", "development_recall",
        "development_recall_minus_c1", "minimum_validation_month_recall",
    ]])
    regularization_summary = pd.DataFrame(regularization_report["summary"])
    display(regularization_summary.loc[regularization_summary.model_or_selection.isin(regularization_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    solver_rows = []
    for entry in regularization_report["monthly_training"]:
        new_fits = [entry["models"][c] for c in regularization_report["new_candidate_order"]]
        control = entry["models"]["logistic_load_regularization_c1"]["c1_reproduction"]
        solver_rows.append({"月份": entry["month"], "全部收斂": all(fit["converged"] for fit in new_fits),
            "最大實際迭代": max(fit["n_iter_max"] for fit in new_fits),
            "C1分數最大差": control["max_abs_score_difference"], "C1迭代相同": control["n_iter_unchanged"]})
    print("收斂與 C1 重現檢查；不是效能分數：")
    display(pd.DataFrame(solver_rows))
    display(pd.DataFrame([regularization_report["suitability"]]))
else:
    print("尚無正則化報告；需要時執行 scripts/tune_logistic_regularization.py。")


## 23. 十二月與其歷史驗證月份的漏報

本節只讀 `december_miss_diagnostics.json`，不重訓、不重選模型、不新增圖表。比較十二月原先由歷史選出的 XGBoost 地理組，以及 Logistic 負載組 C=1、C=0.01。

七、八、九月使用各月初訓練的既有模型，只把 2017-12-01 前已送達的結果納入歷史診斷。每日完整批次先排名、使用 floor(10% × 訂單數) 個提醒名額，結果尚未知的訂單仍占排名與名額。十二月全部結果只供開發回測分析，不能用來改選模型；原保留測試未重新評估。

路線、承諾天數、完整批次排名、歷史支持量與承諾餘裕都只描述關聯。小群組保留但標示樣本不足；三模型共同漏報和互相補回只是錯誤重疊，不是已驗證的合併提醒策略。完整方法與實際結論見 `docs/STAGE4_DECEMBER_MISS_DIAGNOSTICS.md`。


In [ ]:
DECEMBER_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "december_miss_diagnostics.json"
if DECEMBER_DIAGNOSTICS_PATH.is_file():
    december_diagnostics = json.loads(DECEMBER_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
    print("每日上限：", december_diagnostics["daily_cap"], "新增 fit：", december_diagnostics["new_fit_count"])
    print("十二月原選模型：", december_diagnostics["frozen_selection_context"]["selected_candidate"])
    december_rows, overlap_rows = [], []
    for month, entry in december_diagnostics["months"].items():
        batch = entry["complete_batch"]
        for candidate, model in entry["models"].items():
            counts = model["known_outcome_diagnostics"]["aggregate"]["overall"]
            december_rows.append({"月份": month, "角色": entry["role"], "模型": candidate,
                "完整訂單": batch["rows"], "已知結果": batch["known_rows"], "提醒容量": batch["capacity"],
                "未知提醒": model["unknown_outcome_alerts"]["alerts"],
                **{key: counts[key] for key in ("true_positives", "false_positives", "false_negatives", "recall")}})
        overlap_rows.append({"月份": month, "已知延遲": entry["delayed_overlap"]["delayed_rows"],
            "三模型共同漏報": entry["delayed_overlap"]["all_models_miss"]})
    display(pd.DataFrame(december_rows))
    print("共同漏報僅供診斷，不能當作合併模型成績：")
    display(pd.DataFrame(overlap_rows))
    december_selected = december_diagnostics["months"]["2017-12"]["models"]["xgboost_engineering_geo"]
    december_aggregate = december_selected["known_outcome_diagnostics"]["aggregate"]
    for grouping in ("state_route", "promise_days", "route_history_support", "promise_margin"):
        print("十二月 XGBoost 分組：", grouping)
        display(pd.DataFrame(december_aggregate["groups"][grouping]).head(10))
    print("排名區間使用完整每日批次分母：")
    display(pd.DataFrame(december_aggregate["daily_rank_bins"]))
else:
    print("尚無診斷報告；需要時執行 scripts/diagnose_december_misses.py。")


## 24. 路線 × 承諾天數 × 近期負載變化

本節只讀 `route_load_interactions.json` 和 `route_load_diagnostics.json`，不啟動訓練或新增圖表。固定三個基底：Logistic 負載組 C=1、C=0.01，以及 XGBoost 地理組；每個基底分別新增近期趨勢（13 個數值欄）及交互特徵（再加 4 個數值、2 個類別欄），共 48 個新月份模型。

接單趨勢使用核准前 7／14 天對前一同長視窗，納入所有有效核准訂單，不按最終狀態、是否送達或延遲標記篩選。它是負載代理，不能當作實際物流容量。當日完整批次先依分數排名並使用每日 floor(10% × 訂單數) 名額，再篩已知結果計算歷史指標。

三個歷史月最差召回、合計 TP、AP、固定順序沿用原選模規則；30% 只作適用判定。十二月、二月、四月皆是反覆看過的開發回測，不能按這些成績改選；原保留期間不重新評分。完整方法與實際結果見 `docs/STAGE4_ROUTE_LOAD_INTERACTIONS.md`。


In [ ]:
ROUTE_LOAD_PATH = PROJECT_ROOT / "reports" / "stage4" / "route_load_interactions.json"
ROUTE_LOAD_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "route_load_diagnostics.json"
if ROUTE_LOAD_PATH.is_file():
    route_load_report = json.loads(ROUTE_LOAD_PATH.read_text(encoding="utf-8"))
    print("每日上限：", route_load_report["daily_cap"], "新增 fit：", route_load_report["new_fit_count"])
    paired_rows = [{"基底": item["backbone"], **item["development_recall"]}
                   for item in route_load_report["paired_comparisons"]]
    print("固定基底下的開發回測召回；不可按開發結果選模：")
    display(pd.DataFrame(paired_rows))
    route_load_summary = pd.DataFrame(route_load_report["summary"])
    display(route_load_summary.loc[route_load_summary.model_or_selection.isin(route_load_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "validation_folds_below_30_percent",
    ]])
    print("適用判定與開發／獨立測試區分：")
    display(pd.DataFrame([route_load_report["suitability"]]))
    if ROUTE_LOAD_DIAGNOSTICS_PATH.is_file():
        route_load_diagnostics = json.loads(ROUTE_LOAD_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
        paired_month_rows = []
        for month, entry in route_load_diagnostics["months"].items():
            for family, comparison in entry["families"].items():
                for candidate, model in comparison["models"].items():
                    paired_month_rows.append({"月份": month, "角色": entry["role"], "基底": family, "模型": candidate,
                        **{key: model["overall"][key] for key in ("true_positives", "false_positives", "false_negatives", "recall")}})
        print("歷史七／八／九月與十二月開發回測的固定配對：")
        display(pd.DataFrame(paired_month_rows))
else:
    print("尚無實驗報告；需要時執行 scripts/train_route_load_interactions.py。")


## 25. 逾期未送達訂單的 7／14 天堆積變化

本節只讀 `backlog_changes.json` 與 `backlog_diagnostics.json`，不重訓或新增圖表。固定 Logistic 負載 C=1、C=0.01、XGBoost 地理三個原基底，分別新增 7 天、14 天及合併堆積變化，共 72 個新月份模型。

每個快照 s 都只計核准在 [s−90天,s)、原承諾已到期且送達時間未知或 ≥s 的訂單；過去快照不使用後來才知道的配送結果。每個群組／時間差新增當時堆積件數、件數差與 log1p 成長；取消時間不可得、90 天視窗移動也會減少件數，下降不能直接視為物流改善。

每日完整批次先取 floor(10% × 訂單數)，再以已知配送結果計算歷史指標。選模只用三個歷史月的最低召回、TP、AP、固定順序；開發月份已反覆看過，不能當成獨立測試。原保留期間未重新評分，詳細結果見 `docs/STAGE4_BACKLOG_CHANGES.md`。


In [ ]:
BACKLOG_PATH = PROJECT_ROOT / "reports" / "stage4" / "backlog_changes.json"
BACKLOG_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "backlog_diagnostics.json"
if BACKLOG_PATH.is_file():
    backlog_report = json.loads(BACKLOG_PATH.read_text(encoding="utf-8"))
    print("每日上限：", backlog_report["daily_cap"], "新增 fit：", backlog_report["new_fit_count"])
    print("各固定基底的合併開發召回，不能按這張表改選模型：")
    display(pd.DataFrame([{"基底":p["backbone"], **p["development_recall"]} for p in backlog_report["paired_comparisons"]]))
    backlog_summary = pd.DataFrame(backlog_report["summary"])
    display(backlog_summary.loc[backlog_summary.model_or_selection.isin(backlog_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "validation_folds_below_30_percent",
    ]])
    display(pd.DataFrame([backlog_report["suitability"]]))
    if BACKLOG_DIAGNOSTICS_PATH.is_file():
        backlog_diagnostics = json.loads(BACKLOG_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
        backlog_month_rows = []
        for month,entry in backlog_diagnostics["months"].items():
            for family,comparison in entry["families"].items():
                for candidate,model in comparison["models"].items():
                    backlog_month_rows.append({"月份":month,"角色":entry["role"],"基底":family,"模型":candidate,
                        **{k:model["overall"][k] for k in ("true_positives","false_positives","false_negatives","recall")}})
        print("七、八、九月歷史診斷與十二月開發回測：")
        display(pd.DataFrame(backlog_month_rows))
else:
    print("尚無堆積變化報告；需要時執行 scripts/train_backlog_changes.py。")


## 26. 七、九月退步路線與提醒名額交換

只重播既有模型，沒有重訓或重新選模。2017年七、九月為十二月選模時的歷史驗證，結果須在2017-12-01前實際已送達；未知結果仍競爭每日10%名額。路線名額變化是每日邊際帳務，沒有逐筆 donor／recipient 配對或因果結論。

下表查看退步群組、每日同路線／跨路線變動及新增漏報的排名落點。所有路線與小樣本旗標保留於[完整報告](../reports/stage4/backlog_route_exchanges.json)，解讀與重現見[方法文件](../docs/STAGE4_BACKLOG_ROUTE_EXCHANGES.md)。本輪不新增圖表；主開發召回仍25.46%，歷史下限19.17%，適用要求尚未達成。


In [ ]:
ROUTE_EXCHANGE_PATH = PROJECT_ROOT / "reports" / "stage4" / "backlog_route_exchanges.json"
if ROUTE_EXCHANGE_PATH.is_file():
    route_exchange_report = json.loads(ROUTE_EXCHANGE_PATH.read_text(encoding="utf-8"))
    assert route_exchange_report["new_fit_count"] == 0
    assert route_exchange_report["new_model_selection"] is False
    overview, route_losses, exchange_days, lost_ranks = [], [], [], []
    for month, entry in route_exchange_report["months"].items():
        summary = entry["summary"]
        overview.append({"月份": month, "角色": entry["role"],
                         **{k: summary[k] for k in ("known_rows", "unknown_rows", "old_alerts", "new_alerts",
                            "old_true_positives", "new_true_positives", "recovered_delays", "lost_delays",
                            "old_recall", "new_recall")},
                         **entry["allocation_summary"]})
        for row in entry["routes"]:
            if row["net_TP"] < 0:
                route_losses.append({"月份": month, **row})
        for row in sorted(entry["days"], key=lambda row: (row["net_TP"], row["day"]))[:3]:
            exchange_days.append({"月份": month, **row})
        lost = next(row for row in entry["rank_movements"]["cohorts"] if row["cohort"] == "lost_delays")
        lost_ranks.append({"月份": month, "新增漏報": lost["rows"], **lost["new_rank_bins"]})
    print("完整日批次先排名，再篩選已知結果；兩月均為歷史驗證：")
    display(pd.DataFrame(overview))
    print("所有淨TP下降路線，保留樣本門檻旗標；不能按表直接加固定路線配額：")
    route_loss_table = pd.DataFrame(route_losses).sort_values(["月份", "net_TP", "state_route"])
    display(route_loss_table[["月份", "state_route", "known_rows", "positive_rows", "old_alerts", "new_alerts",
                             "recovered_delays", "lost_delays", "net_TP", "net_FP", "minimum_sample_met"]])
    print("各月淨TP最差的三個日期，日容量與名額交換仍守恆：")
    display(pd.DataFrame(exchange_days)[["月份", "day", "full_batch_rows", "capacity",
                                       "old_true_positives", "new_true_positives", "dropped_alerts",
                                       "within_route_exchanges", "cross_route_slots"]])
    print("新增漏報在新模型完整日排名的落點：")
    display(pd.DataFrame(lost_ranks))
else:
    print("尚無名額交換報告；執行 scripts/diagnose_backlog_route_exchanges.py，只重播、不訓練。")


## 27. 相同83／101欄下的分類與每日排序目標

固定本輪特徵、deadline_wait0訓練時鐘、核心樹參數與完整每日10%預算，比較XGBoost分類與每日rank:ndcg。16個分類月份版本重播，16個排序版本新訓練；另63個候選的舊成績從綁定雜湊的原報告沿用，沒有重播其504個版本。新分數只包含四個成對候選。

排序訓練使用成熟列的核准日群組、固定topk20，不等於每天剛好10%的提醒切點；sigmoid margin只是未校準的有界分數。沒有開發月eval_set、early stopping或依結果改參數。這些開發月份已被反覆檢視，不能當成新獨立測試。

詳細方法與失敗結果見[分類與排序實驗](../docs/STAGE4_BACKLOG_OBJECTIVES.md)，整體歷程見[專案進度](../docs/PROJECT_PROGRESS.md)。下列只新增表格，Dashboard仍暫緩。


In [ ]:
OBJECTIVE_PATH = PROJECT_ROOT / "reports" / "stage4" / "backlog_objectives.json"
OBJECTIVE_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "backlog_objective_diagnostics.json"
if OBJECTIVE_PATH.is_file():
    objective_report = json.loads(OBJECTIVE_PATH.read_text(encoding="utf-8"))
    print("本輪新fit：", objective_report["new_fit_count"], "分類版本重播：", objective_report["replayed_classifier_version_count"])
    objective_summary = pd.DataFrame(objective_report["summary"])
    objective_names = objective_report["matched_candidate_order"] + ["daily_rankers", "matched_four", "primary_without_dispatch"]
    print("相同每日10%預算下的固定候選與歷史選模結果：")
    display(objective_summary.loc[objective_summary.model_or_selection.isin(objective_names), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "recall", "minimum_validation_month_recall", "validation_folds_below_30_percent"]])
    objective_month_rows = []
    for fold in objective_report["folds"]:
        for candidate in objective_report["matched_candidate_order"]:
            record = fold["experiment"]["models"][candidate]
            for month, part in record["validation_by_month"].items():
                metric = part["known_outcome_metrics"]
                objective_month_rows.append({"選模月": fold["name"], "月份": month, "角色": "historical_validation",
                    "模型": candidate, "TP": metric["confusion_matrix"][1][1], "召回": metric["recall"]})
            metric = record["development_evaluation"]
            objective_month_rows.append({"選模月": fold["name"], "月份": fold["name"], "角色": "development_evaluation",
                "模型": candidate, "TP": metric["confusion_matrix"][1][1], "召回": metric["recall"]})
    display(pd.DataFrame(objective_month_rows))
    if OBJECTIVE_DIAGNOSTICS_PATH.is_file():
        objective_diagnostics = json.loads(OBJECTIVE_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
        regression_rows, route_change_rows = [], []
        for phase in objective_diagnostics["phases"].values():
            if phase["fold"] != "2017-12" or phase["month"] not in ["2017-07", "2017-09"]:
                continue
            cohort = phase["original_regression_cohort"]
            regression_rows.append({"月份": phase["month"], "原新增漏報": cohort["cohort_rows"],
                                    **{c: m["in_top10"] for c, m in cohort["models"].items()}})
            for package, comparison in phase["objectives"].items():
                for row in comparison["routes"]:
                    if row["net_TP"] != 0:
                        route_change_rows.append({"月份": phase["month"], "特徵組": package, **row})
        print("原本因加入堆積特徵而新增的漏報，多少重新進前10%；仍須一起看其他新增漏報：")
        display(pd.DataFrame(regression_rows))
        print("七九月所有TP有變化路線，正負皆保留，不能按此直接加固定配額：")
        display(pd.DataFrame(route_change_rows)[["月份", "特徵組", "state_route", "known_rows", "positive_rows",
            "recovered_delays", "lost_delays", "net_TP", "net_FP", "minimum_sample_met"]])
else:
    print("尚無本輪報告；需要時依文件執行 scripts/compare_backlog_objectives.py。")


## 28. 固定每日排序設定，只比較L2正則化

固定83／101欄、成熟訓練訂單與日群組、rank:ndcg／topk20、400棵樹與其他參數，只比較reg_lambda=1／10／100。10沿用既有模型；1及100各兩種特徵、八個月份，共32個新版本。每日提醒仍按完整批次floor(10%×N)，未知結果先競爭名額，歷史指標只計cutoff前實際送達者。

九月SP→SP的101欄λ10基準為188提醒、8TP、180FP。下表同時檢查原新增誤報移出、失去延遲補回，以及其他路線得失；不按單一路線事後改選。三個E月都已反覆檢視，原保留期間不重新評分，原RF維持。完整方法與結果見[L2實驗文件](../docs/STAGE4_RANK_REGULARIZATION.md)，本節只有表格。


In [ ]:
RANK_L2_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_regularization.json"
RANK_L2_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_regularization_diagnostics.json"
if RANK_L2_PATH.is_file():
    rank_l2_report = json.loads(RANK_L2_PATH.read_text(encoding="utf-8"))
    print("新增fit：", rank_l2_report["new_fit_count"], "原配對版本重播：", rank_l2_report["replayed_previous_version_count"])
    rank_l2_paired = []
    for package in rank_l2_report["paired_comparisons"]:
        for strength, candidate in package["candidates_by_l2"].items():
            rank_l2_paired.append({"特徵組": package["feature_package"], "L2": strength, "候選": candidate,
                "開發TP": package["development_true_positives"][strength],
                "開發召回": package["development_recall"][strength], "TP差相對10": package["net_TP_vs_l2_10"][strength]})
    print("相同10%預算下的固定候選，保留所有強度：")
    display(pd.DataFrame(rank_l2_paired))
    rank_l2_summary = pd.DataFrame(rank_l2_report["summary"])
    display(rank_l2_summary.loc[rank_l2_summary.model_or_selection.isin(rank_l2_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "recall", "minimum_validation_month_recall", "validation_folds_below_30_percent"]])
    if RANK_L2_DIAGNOSTICS_PATH.is_file():
        rank_l2_diagnostics = json.loads(RANK_L2_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
        september = rank_l2_diagnostics["phases"]["2017-12/2017-09/validation"]
        local_rows, other_routes, cohort_rows = [], [], []
        for candidate, comparison in september["classification_references"].items():
            route = next(row for row in comparison["routes"] if row["state_route"] == "SP->SP")
            local_rows.append({"排序候選": candidate,
                **{key: route[key] for key in ("known_rows", "positive_rows", "old_alerts", "new_alerts",
                    "old_true_positives", "new_true_positives", "old_false_positives", "new_false_positives")}})
        print("九月歷史驗證，SP→SP分類參照與各L2排序版本：")
        display(pd.DataFrame(local_rows))
        for name, cohort in september["regression_cohorts"]["cohorts"].items():
            local = next(row for row in cohort["routes"] if row["state_route"] == "SP->SP")
            for scope, value in (("全部路線", cohort), ("SP->SP", local)):
                for candidate, counts in value["models"].items():
                    cohort_rows.append({"固定群體": name, "範圍": scope, "原件數": value["cohort_rows"],
                        "候選": candidate, "前10%": counts["in_top10"], "前10%外": counts["out_of_top10"]})
        print("新增準時提醒看移出數，失去延遲看補回數；仍須一起看新增的其他錯誤：")
        display(pd.DataFrame(cohort_rows))
        for candidate, comparison in september["regularization"].items():
            for route in comparison["routes"]:
                if route["net_TP"] != 0:
                    other_routes.append({"候選": candidate, **route})
        print("九月所有TP變動路線，正負皆保留；小樣本仍標記：")
        display(pd.DataFrame(other_routes)[["候選", "state_route", "known_rows", "positive_rows",
            "recovered_delays", "lost_delays", "net_TP", "net_FP", "minimum_sample_met"]])
else:
    print("尚無L2報告；需要時依文件執行 scripts/compare_rank_regularization.py。")


## 29. 固定L2＝100，比較排序配對焦點

固定83／101欄、成熟訓練訂單與日群組、rank:ndcg、400棵樹與其他參數，只比較原生topk配對焦點10／20／40。20沿用既有版本；10及40各兩種特徵、八個月份，共32次新fit。固定k是訓練配對位置，並不是每天10%的切點；提醒仍按完整批次floor(10%×N)，未知結果先競爭名額，再遮罩答案。

七、九月以2017-12-01之前實際送達結果作歷史驗證，分別檢查「兩月均不差於本輪k20」及「七月恢復早先L2＝10、九月保留L2＝100水準」，兩項只作描述，不加入選模。75候選按原歷史規則選出的主方案，開發TP由418降至413、召回25.46%降至25.15%；不能按已看過的E月改回或稱為新測試改善。原保留期間未重新評分，原RF維持。詳見[配對焦點實驗](../docs/STAGE4_RANK_PAIR_FOCUS.md)，本節只有表格。


In [ ]:
RANK_FOCUS_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_pair_focus.json"
RANK_FOCUS_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_pair_focus_diagnostics.json"
if RANK_FOCUS_PATH.is_file():
    rank_focus_report = json.loads(RANK_FOCUS_PATH.read_text(encoding="utf-8"))
    print("新增fit：", rank_focus_report["new_fit_count"], "原配對版本重播：", rank_focus_report["replayed_previous_version_count"])
    rank_focus_paired = []
    for package in rank_focus_report["paired_comparisons"]:
        for focus, candidate in package["candidates_by_focus"].items():
            rank_focus_paired.append({"特徵組": package["feature_package"], "配對k": int(focus), "候選": candidate,
                "開發TP": package["development_true_positives"][focus],
                "開發召回": package["development_recall"][focus], "TP差相對k20": package["net_TP_vs_top20"][focus]})
    print("固定候選的三個已檢視開發月，所有k均保留：")
    display(pd.DataFrame(rank_focus_paired))
    print("七、九月的兩個事前兼顧目標，未用於選模：")
    display(pd.DataFrame(rank_focus_report["compatibility_checks"]))
    rank_focus_summary = pd.DataFrame(rank_focus_report["summary"])
    display(rank_focus_summary.loc[rank_focus_summary.model_or_selection.isin(rank_focus_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "recall", "minimum_validation_month_recall", "validation_folds_below_30_percent"]])
    if RANK_FOCUS_DIAGNOSTICS_PATH.is_file():
        rank_focus_diagnostics = json.loads(RANK_FOCUS_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
        rank_focus_routes = []
        for month in ("2017-07", "2017-09"):
            phase = rank_focus_diagnostics["phases"][f"2017-12/{month}/validation"]
            for candidate, comparison in phase["pair_focus"].items():
                for route in comparison["routes"]:
                    if route["net_TP"] != 0:
                        rank_focus_routes.append({"歷史月": month, "候選": candidate, **route})
        print("七、九月所有TP變動路線，補抓與失去同時列出；小樣本保留旗標：")
        display(pd.DataFrame(rank_focus_routes)[["歷史月", "候選", "state_route", "known_rows", "positive_rows",
            "old_alerts", "new_alerts", "recovered_delays", "lost_delays", "net_TP", "net_FP", "minimum_sample_met"]])
else:
    print("尚無配對焦點報告；需要時依文件執行 scripts/compare_rank_pair_focus.py。")


## 30. 固定模型，按每日訂單量診斷七月與十二月

本步只讀前輪已凍結分數，0次fit、0次模型重播、0次選模。按完整日批次N分成1–99／100–199／200–399／400+，提醒仍取floor(0.10×N)。七月是2017-12 fold的歷史驗證，只計實際送達嚴格早於2017-12-01的答案；十二月是已檢視開發月的事後診斷，並非新測試。

同101欄k20→10的十二月TP68→65，與主方案83欄分類→101欄排序TP70→65分開列出；後者還變更特徵、目標與正則化，不能全歸因於k。固定k只對訓練成熟query中的前段位置配對，與推論完整日容量不同；位置覆蓋不是實際配對權重。一天的分箱只供描述，數值接近10%也不能證明改善原因。詳見[每日訂單量診斷](../docs/STAGE4_RANK_BATCH_SIZE.md)，本節只有表格。


In [ ]:
RANK_BATCH_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_batch_size_diagnostics.json"
if RANK_BATCH_PATH.is_file():
    rank_batch_report = json.loads(RANK_BATCH_PATH.read_text(encoding="utf-8"))
    rank_batch_base = "xgboost_rank_backlog_7_14d_l2_100"
    rank_batch_names = {20: rank_batch_base, 10: rank_batch_base + "_topk_10", 40: rank_batch_base + "_topk_40"}
    rank_batch_rows, rank_batch_changes, rank_batch_alignment, rank_batch_days = [], [], [], []
    for phase in rank_batch_report["phases"].values():
        for group in phase["volume_bins"]:
            base = group["models"][rank_batch_base]
            rank_batch_rows.append({"月份": phase["month"], "用途": phase["role"], "每日N": group["volume_bin"],
                "天數": base["days"], "完整訂單": base["full_rows"], "已知延遲": base["known_delayed_rows"],
                "提醒容量": base["capacity"], **{f"TP_k{k}": group["models"][candidate]["TP"] for k, candidate in rank_batch_names.items()}})
        for key, comparison in phase["contrasts"].items():
            for group in comparison["by_volume_bin"]:
                rank_batch_changes.append({"月份": phase["month"], "比較": key,
                    "比較類型": comparison["comparison"]["comparison_kind"], "每日N": group["volume_bin"],
                    **{field: group[field] for field in ("old_true_positives", "new_true_positives", "recovered_delays", "lost_delays", "net_TP", "net_FP")}})
        for k, alignment in phase["k_alignment"].items():
            rank_batch_alignment.append({"月份": phase["month"], "k": int(k), "批次天數": alignment["days"],
                "k小於容量天數": alignment["relation"]["k_lt_capacity"]["days"],
                "k等於容量天數": alignment["relation"]["k_eq_capacity"]["days"],
                "k大於容量天數": alignment["relation"]["k_gt_capacity"]["days"]})
        for day in phase["contrasts"][rank_batch_names[10]]["daily"]:
            if day["net_TP"] != 0:
                rank_batch_days.append({"日期": day["day"], **{field: day[field] for field in (
                    "full_batch_rows", "old_alerts", "old_true_positives", "new_true_positives", "recovered_delays", "lost_delays", "net_TP")}})
    print("101欄固定L2=100，保留全部四個日量分箱；一日分箱僅供描述：")
    display(pd.DataFrame(rank_batch_rows))
    print("所有同特徵k對照與十二月主方案混合改動分開列示，正負均保留：")
    display(pd.DataFrame(rank_batch_changes))
    print("固定k與完整每日10%提醒名額的數值關係，並非訓練／推論母體相同：")
    display(pd.DataFrame(rank_batch_alignment))
    print("101欄k20→10所有TP變動日期：")
    display(pd.DataFrame(rank_batch_days))
    rank_batch_training = []
    for month, entry in rank_batch_report["training_query_alignment"]["months"].items():
        summary = entry["summary"]
        for k, focus in summary["focus"].items():
            rank_batch_training.append({"模型月份": month, "k": int(k), "成熟訓練列": summary["training_rows"],
                "訓練日群組": summary["queries"], "混合標籤日": summary["mixed_label_queries"],
                "query列數中位": summary["query_size_median"], "query列數最大": summary["query_size_max"],
                "名義位置覆蓋率": focus["focus_position_fraction"],
                "混合query中k等於10%天數": focus["alignment"]["mature_query"]["mixed_label_queries"]["configured_k_relation"]["equal"]["queries"]})
    print("訓練query是當時成熟子集；覆蓋率為sum(min(k,q))/sum(q)，不是實際pairs或權重：")
    display(pd.DataFrame(rank_batch_training))
else:
    print("尚無每日訂單量診斷；需要時依文件執行 scripts/diagnose_rank_batch_size.py。")


## 31. 固定101欄與L2＝100，只比較topk與mean配對方式

固定成熟訓練訂單、每日query、前處理、數值參數10及其他模型設定，只將lambdarank_pair_method由topk改為mean。八個mean版本新訓練、八個既有topk10版本重播；75個既有候選指標凍結沿用，再加入一個mean候選，仍按原歷史規則選模。10在mean代表每列取樣配對數，不是前10名或動態每日10%；normalization設定不變，但內建行為依method不同，不能稱為同計算量或純切點效果。

所有評估維持完整每日批次floor(0.10×N)，未知答案先競爭名額再遮罩。歷史驗證只計實際送達嚴格早於各fold截止的答案，三個開發月已反覆檢視，不能作為新未知測試。mean固定候選開發TP339→226、召回20.65%→13.76%；主76候選未選mean，仍413TP／25.15%。原RF與保留期間維持。詳見[topk與mean實驗](../docs/STAGE4_RANK_PAIR_METHOD.md)，本節只有表格。


In [ ]:
RANK_METHOD_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_pair_method.json"
RANK_METHOD_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "rank_pair_method_diagnostics.json"
if RANK_METHOD_PATH.is_file():
    rank_method_report = json.loads(RANK_METHOD_PATH.read_text(encoding="utf-8"))
    rank_method_summary = pd.DataFrame(rank_method_report["summary"])
    print("新增mean訓練：", rank_method_report["fit_count_this_execution"],
          "既有topk版本重播：", rank_method_report["replayed_previous_version_count"])
    display(rank_method_summary.loc[rank_method_summary.model_or_selection.isin(
        rank_method_report["matched_candidate_order"] + list(rank_method_report["selection_groups"])), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "recall", "minimum_validation_month_recall", "validation_folds_below_30_percent"]])
    print("事前已知的七月／九月歷史、十二月開發參照只描述、不選模：")
    display(pd.DataFrame(rank_method_report["compatibility_checks"]))
    if RANK_METHOD_DIAGNOSTICS_PATH.is_file():
        rank_method_diagnostics = json.loads(RANK_METHOD_DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
        rank_method_phases, rank_method_volume, rank_method_routes = [], [], []
        for key, phase in rank_method_diagnostics["phases"].items():
            comparison = phase["pair_method"]
            summary = comparison["summary"]
            rank_method_phases.append({"階段": key, "用途": phase["role"],
                **{field: summary[field] for field in ("positive_rows", "old_true_positives", "new_true_positives",
                    "recovered_delays", "lost_delays", "net_TP", "net_FP")}})
            for group in comparison["volume_bins"]:
                rank_method_volume.append({"階段": key, "每日N": group["volume_bin"], "天數": group["days"],
                    "完整訂單": group["full_batch_rows"], "容量": group["capacity"],
                    **{field: group[field] for field in ("old_true_positives", "new_true_positives", "recovered_delays", "lost_delays", "net_TP")}})
            if key in ("2017-12/2017-07/validation", "2017-12/2017-09/validation",
                       "2017-12/2017-12/evaluation", "2018-02/2018-02/evaluation", "2018-04/2018-04/evaluation"):
                for route in comparison["routes"]:
                    if route["net_TP"] != 0:
                        rank_method_routes.append({"階段": key, **route})
        print("全部12個歷史／開發phase各自列示；重複月份有不同答案截止，不能當獨立樣本相加：")
        display(pd.DataFrame(rank_method_phases))
        print("全部四個日量分箱均保留，一日／空分箱僅供描述：")
        display(pd.DataFrame(rank_method_volume))
        print("關注月份所有非零TP路線，補抓與新漏報同時保留；小樣本仍有旗標：")
        display(pd.DataFrame(rank_method_routes)[["階段", "state_route", "known_rows", "positive_rows",
            "old_alerts", "new_alerts", "old_true_positives", "new_true_positives",
            "recovered_delays", "lost_delays", "net_TP", "net_FP", "minimum_sample_met"]])
else:
    print("尚無配對方式報告；需要時依文件執行 scripts/compare_rank_pair_method.py。")


## 32. 物流訊號與預測時點盤點

本步0次訓練、0次新模型預測，盤點9份CSV欄位與事件時計。作業與歷史支持分析只用核准、事件及查詢都嚴格早於2018-05-01的資料；整份檔案只檢查schema、列數、大小與SHA，不使用原保留期間答案。73,254筆有效核准母體沒有按最終狀態或最終送達篩選，與既有模型母體不同，不能直接比較召回。

事件特徵只看事件<t；送達<=t保守排除提醒。已過承諾但未送達是逾期監測，與尚未過承諾的提前預測分開。carrier事件後1ns只是嚴格時計的排序記號，不代表真實系統奈秒可用；CSV沒有入庫時間與歷史版本，事件先後只能在及時入庫等假設下代表可用性。

建議下一輪維持核准時點與每日10%預算，101→104欄，只新增主賣家過去90天交運事件數、核准至交運耗時p50/p90，小於5個有效事件時耗時留缺失。這是研究方案，還沒有訓練或召回成績；未交運負載保留作另一個獨立候選。詳見[物流訊號與時點盤點](../docs/STAGE4_LOGISTICS_AVAILABILITY.md)，本節只有表格。


In [ ]:
LOGISTICS_AVAILABILITY_PATH = PROJECT_ROOT / "reports" / "stage4" / "logistics_availability.json"
if LOGISTICS_AVAILABILITY_PATH.is_file():
    logistics_report = json.loads(LOGISTICS_AVAILABILITY_PATH.read_text(encoding="utf-8"))
    print("本輪新訓練／新模型預測：", logistics_report["new_training_fits"], logistics_report["new_model_predictions"])
    print("原始schema全檔只作欄位、列數、大小與雜湊盤點：")
    display(pd.DataFrame(logistics_report["raw_sources"])[["file", "rows", "columns", "bytes"]])
    print("各預測時點的可知／未知與新任務限制：")
    display(pd.DataFrame(logistics_report["prediction_time_inventory"]))
    logistics_sequences, logistics_checkpoints = [], []
    for period, profile in logistics_report["operational_profiles"].items():
        logistics_sequences.append({"核准範圍": period, "有效核准數": profile["orders"],
            "非負交運事件": profile["valid_observed_handover_events"], **profile["milestone_sequence_anomalies"]})
        points = {f"approval+{day}d": value for day,value in profile["approval_checkpoints"].items()}
        points["carrier+1ns"] = profile["handover_checkpoint"]
        points["carrier+3d+1ns"] = profile["transit_checkpoint_carrier_plus_3d"]
        for name, point in points.items():
            logistics_checkpoints.append({"核准範圍": period, "時點": name,
                **{field: point[field] for field in ("observable_queries_before_cutoff", "eligible_for_reminder_proxy",
                    "already_overdue_pending_among_generic_eligible", "eligible_for_advance_reminder_proxy",
                    "customer_delivery_equal_query_conservative_suppression")}})
    print("時序異常不可補成0耗時；多種異常可能重疊，不能相加：")
    display(pd.DataFrame(logistics_sequences))
    print("一般未送達監測／已逾期／仍可提前提醒分開；不同時點分母不能互相比召回：")
    display(pd.DataFrame(logistics_checkpoints))
    logistics_history = []
    for month, summary in logistics_report["handover_history_feasibility"]["by_approval_month"].items():
        for group, support in summary["groups"].items():
            logistics_history.append({"核准月": month, "分組": group, "查詢訂單": summary["query_rows"],
                **{field: support[field] for field in ("no_past_event_rows", "at_least_five_event_rows",
                    "at_least_five_event_fraction", "support_median")}})
    print("過去90天支持數，區間為[t−90天,t)；只表示可行性，不是模型效果：")
    display(pd.DataFrame(logistics_history))
    print("既有特徵／已有分支／尚未實作訊號分開：")
    display(pd.DataFrame(logistics_report["candidate_signal_inventory"]))
else:
    print("尚無物流盤點；需要時執行 scripts/audit_logistics_availability.py。")


## 33. 原101欄與主賣家交運歷史104欄

固定核准時點、L2＝100、topk10、每日完整批次floor(10%×N)，只新增主賣家過去90天有效交運事件數及核准至交運耗時p50／p90。每張訂單只查[t−90天,t)事件，排除本單及已知矛盾；不足5件時分位數留缺失，並沿用訓練資料補值。這是訂單級交運代理，並非每個包裹速度，仍假設事件及時入庫。

八個104欄模型新訓練，八份101欄舊模型重放；舊76候選的指標凍結後加1候選，按原歷史規則選模。先列固定候選對照，再列77候選主方案，避免混淆。2017-12／2018-02／2018-04均已反覆使用的開發回測，未重新評估原保留測試；本節只有表格。詳見[交運歷史比較](../docs/STAGE4_HANDOVER_HISTORY.md)。


In [ ]:
HANDOVER_PATH = PROJECT_ROOT / "reports" / "stage4" / "handover_history.json"
HANDOVER_DIAGNOSTICS = PROJECT_ROOT / "reports" / "stage4" / "handover_history_diagnostics.json"
if HANDOVER_PATH.is_file() and HANDOVER_DIAGNOSTICS.is_file():
    handover = json.loads(HANDOVER_PATH.read_text(encoding="utf-8"))
    handover_diag = json.loads(HANDOVER_DIAGNOSTICS.read_text(encoding="utf-8"))
    print("只改三個特徵，固定候選的三個開發月：")
    handover_rows = []
    for month, pair in handover["paired_comparisons"]["by_fold"].items():
        for name, item in pair.items():
            metric = item["development"]
            handover_rows.append({"開發月": month, "特徵組": name, "TP": metric["confusion_matrix"][1][1],
                "召回率": metric["recall"], "歷史三月下限": item["minimum_validation_month_recall"]})
    display(pd.DataFrame(handover_rows))
    print("各候選／歷史主方案摘要（固定候選與主方案分開）：")
    display(pd.DataFrame([row for row in handover["summary"] if row["model_or_selection"] in
        handover["matched_candidate_order"] + ["previous_candidates", "handover_features", "primary_without_dispatch"]]))
    print("七、九月歷史與十二月開發參照，僅描述，未用來改選：")
    display(pd.DataFrame(handover["compatibility_checks"]))
    handover_phase_rows, handover_support_rows, handover_route_rows = [], [], []
    for key, phase in handover_diag["phases"].items():
        contrast = phase["handover_features"]
        handover_phase_rows.append({"階段": key, **contrast["summary"]})
        handover_support_rows.extend({"階段": key, **row} for row in contrast["feature_support_bins"])
        if key in ("2017-12/2017-07/validation", "2017-12/2017-09/validation", "2017-12/2017-12/evaluation"):
            handover_route_rows.extend({"階段": key, **row} for row in contrast["routes"] if row["net_TP"] != 0)
    print("全部12個phase分列，重複歷史月份不能當獨立樣本加總：")
    display(pd.DataFrame(handover_phase_rows))
    print("有／無足夠交運歷史的得失；提醒名額仍按完整日分配：")
    display(pd.DataFrame(handover_support_rows))
    print("關注月份所有非零TP路線（包含改善與退步，小樣本保留旗標）：")
    display(pd.DataFrame(handover_route_rows))
else:
    print("尚無交運歷史報告；需要時依文件執行 scripts/train_handover_history.py 與診斷。")


## 34. 固定模型追查稀疏歷史、長承諾與SP→RJ的交集

本輪只讀凍結的101／104欄分數，不訓練、不重播模型、不改選，也不重新評估原保留測試。核准時點、L2＝100、topk10與完整日floor(10%×N)維持。將歷史0筆／1–4筆／至少5筆、承諾≤30／>30天、SP→RJ／其他路線組成12個互斥格，另列八個重疊視角，檢查原先邊際退步是否真發生在同一批訂單。

「31+」沿用既有分箱，實際為估計承諾天數>30，含30.x天；不是重新搜尋的門檻。所有排名、名額先由完整日決定，再看已知答案與子群。跨格incoming／outgoing是同日總量記帳，同日新增提醒也不代表某張訂單取代了另一張。早期歷史月份保留各自答案截止，開發結果已反覆看過，不能相加當新的未知測試。詳見[固定模型交集診斷](../docs/STAGE4_HANDOVER_SPARSE_DIAGNOSTICS.md)，本節只有表格。


In [ ]:
HANDOVER_SPARSE_PATH = PROJECT_ROOT / "reports" / "stage4" / "handover_sparse_diagnostics.json"
if HANDOVER_SPARSE_PATH.is_file():
    sparse_report = json.loads(HANDOVER_SPARSE_PATH.read_text(encoding="utf-8"))
    print("本輪fit／模型重播／改選：", sparse_report["new_fit_count"],
          sparse_report["saved_versions_replayed_in_this_cli"], sparse_report["new_model_selection"])
    print("全部12個phase，重複歷史月份各自保留原答案截止：")
    display(pd.DataFrame(sparse_report["phase_summary"])[["phase", "role", "positive_rows",
        "old_true_positives", "new_true_positives", "recovered_delays", "lost_delays", "net_TP"]])
    sparse_april = sparse_report["phases"]["2018-04/2018-04/evaluation"]
    print("四月12個互斥格，空格也保留，不能由重疊邊際直接相加：")
    display(pd.DataFrame(sparse_april["intersection_cells"])[["cell", "full_batch_rows", "positive_rows",
        "old_alerts", "new_alerts", "old_true_positives", "new_true_positives",
        "recovered_delays", "lost_delays", "small_sample"]])
    print("四月八個重疊視角分開看：")
    display(pd.DataFrame(sparse_april["focus_masks"])[["mask", "full_batch_rows", "positive_rows",
        "old_true_positives", "new_true_positives", "recovered_delays", "lost_delays", "small_sample"]])
    sparse_rank_rows, sparse_history_rows = [], []
    for key, phase in sparse_report["phases"].items():
        for item in phase["focus_masks"]:
            if item["mask"] in ("history_lt5", "promise31plus_and_SP_RJ", "triple"):
                sparse_history_rows.append({"階段": key, "視角": item["mask"],
                    **{field: item[field] for field in ("positive_rows", "old_true_positives",
                        "new_true_positives", "recovered_delays", "lost_delays", "net_TP", "small_sample")}})
                if key == "2018-04/2018-04/evaluation":
                    for name, rank in item["rank_analysis"]["changed_and_missed_rank_details"].items():
                        sparse_rank_rows.append({"視角": item["mask"], "群體": name, "數量": rank["rows"],
                            "原排名": rank["old_positions"], "新排名": rank["new_positions"],
                            "新排名超過每日名額的距離": rank["off_top10_gap_slots"]["new"]})
    print("四月完整日排名與名額距離，沒有跨模型直接比原始分數：")
    display(pd.DataFrame(sparse_rank_rows))
    print("同規格的較早月份對照；小樣本及零延遲仍保留：")
    display(pd.DataFrame(sparse_history_rows))
    print("四月跨互斥格名額記帳：")
    display(pd.DataFrame([sparse_april["cell_allocation"]["summary"]]))
    print("四月實際路線／原五個承諾分箱／歷史支持組的非零TP得失：")
    display(pd.DataFrame([row for row in sparse_april["observed_route_promise_support"] if row["net_TP"] != 0]))
else:
    print("尚無交集診斷；需要時執行 scripts/diagnose_handover_sparse.py。")


## 本階段留下的成果

- 可重用的訂單特徵、延遲標記、歷史與待件負載建置程式。
- 多月份 OOT、每月重訓、結果成熟篩選與每日 10% 提醒預算。
- 同模型特徵工程比較、成對結果與完整失敗紀錄。
- 預設模型與研究 artifact 分開保留，附完整日批次推論介面。

目前持續改善固定預算召回；先確認歷史月份的適用門檻與後續新期間表現，再進行 Dashboard。開發回測不能當成未知資料成績。
